In [ ]:
%%time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from bs4 import BeautifulSoup
import time
import csv
import re

# Function to check robots.txt
def check_robots(url):
    from urllib.robotparser import RobotFileParser
    from urllib.parse import urlparse

    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    rp.read()
    return rp.can_fetch("*", url)

# Set target URL
base_url = "https://www.junaidjamshed.com/fragrances/for-men.html"

# Check robots.txt
if not check_robots(base_url):
    print("Scraping disallowed by robots.txt")
    exit()

# Setup headless browser with user-agent
options = Options()
options.headless = True
options.add_argument("window-size=1920x1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Start browser
all_data = []
driver = None
try:
    driver = webdriver.Chrome(options=options)
    page_num = 1
    while True:
        url = f"{base_url}?p={page_num}"
        print(f"Scraping page {page_num}...")
        try:
            driver.get(url)
        except WebDriverException as e:
            print("WebDriverException on listing page:", e)
            break
        time.sleep(6)  # Wait for content to load
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        products = soup.select("li.item.product.product-item")
        if not products:
            print("No more products found. Ending pagination.")
            break
        for product in products:
            name = product.select_one("a.product-item-link")
            price = product.select_one("span.price")
            image = product.select_one("img.product-image-photo")
            rating = None
            rating_elem = (
                product.select_one(".rating-result") or
                product.select_one(".product-reviews-summary .rating") or
                product.select_one(".review-ratings") or
                product.select_one(".rating-summary") or
                product.select_one("span[class*='star']")
            )
            if rating_elem:
                style = rating_elem.get('style')
                if style:
                    match = re.search(r'width:\s*(\d+)%', style)
                    if match:
                        percent = int(match.group(1))
                        rating = round(percent / 20, 1)  # 100% = 5.0, 90% = 4.5, etc.
                if not rating:
                    text = rating_elem.get_text(strip=True)
                    match = re.search(r'(\d+\.\d+)', text)
                    if match:
                        rating = float(match.group(1))
            if name and name.has_attr("href"):
                product_url = name["href"]
                try:
                    driver.get(product_url)
                    time.sleep(3)
                    detail_soup = BeautifulSoup(driver.page_source, 'html.parser')
                    # Try to get rating again if not found
                    if not rating:
                        detail_rating_elem = (
                            detail_soup.select_one(".rating-result") or
                            detail_soup.select_one(".product-reviews-summary .rating") or
                            detail_soup.select_one(".review-ratings") or
                            detail_soup.select_one(".rating-summary") or
                            detail_soup.select_one("span[class*='star']")
                        )
                        if detail_rating_elem:
                            style = detail_rating_elem.get('style')
                            if style:
                                match = re.search(r'width:\s*(\d+)%', style)
                                if match:
                                    percent = int(match.group(1))
                                    rating = round(percent / 20, 1)
                            if not rating:
                                text = detail_rating_elem.get_text(strip=True)
                                match = re.search(r'(\d+\.\d+)', text)
                                if match:
                                    rating = float(match.group(1))
                except WebDriverException as e:
                    print(f"WebDriverException fetching rating for {name.get_text(strip=True)}: {e}")
                    break  # Stop processing if driver is dead
                except Exception as e:
                    print(f"Error fetching rating for {name.get_text(strip=True)}: {e}")
                # Return to listing page
                try:
                    driver.get(url)
                    time.sleep(2)
                except WebDriverException as e:
                    print("WebDriverException returning to listing page:", e)
                    break
            all_data.append({
                "name": name.get_text(strip=True) if name else None,
                "price": price.get_text(strip=True) if price else None,
                "image": image['src'] if image and image.has_attr('src') else None,
                "rating": rating
            })
        page_num += 1
except Exception as e:
    print("Error during scraping:", e)
finally:
    if driver:
        try:
            driver.quit()
        except Exception:
            pass

# Save to CSV
csv_filename = "j_fragrances_men.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=["name", "price", "image", "rating"])
    writer.writeheader()
    writer.writerows(all_data)

print(f"Saved {len(all_data)} products to {csv_filename}")


Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
No more products found. Ending pagination.
Saved 103 products to j_fragrances_men.csv
CPU times: total: 2min 13s
Wall time: 36min 26s
